# SmolVLA_RL — Colab Runner

SFT SmolVLA on LIBERO-Plus, then run post-training (GRPO / RS-SFT / flow-DPO)
and evaluation. **The decisive result is the preregistered unseen-task
evaluation in cell 7**; the in-dist / held-out evaluation in cells 5-6 is
exploratory (see the caveat there).

## Prerequisites
- Runtime: **A100** (lower `BATCH_SIZE` on L4/T4)
- Mount Drive to persist checkpoints/logs
- Clone and use the SmolVLA_RL repository from GitHub

## 1. Mount Drive + clone the repo

In [ ]:
from google.colab import drive, userdata
import base64, os, pathlib, subprocess

drive.mount('/content/drive')
REPO_URL = 'https://github.com/Ono-Katsuki/SmolVLA_RL.git'
REPO_DIR = '/content/SmolVLA_RL'
DRIVE_ROOT = '/content/drive/MyDrive/SmolVLA_RL'
pathlib.Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

# A private repo uses GITHUB_TOKEN from Colab Secrets. Never emit the value into commands or logs.
token = userdata.get('GITHUB_TOKEN')
credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy()
env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader',
           GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {credential}')
cmd = ['git', '-C', REPO_DIR, 'pull', '--ff-only'] if pathlib.Path(REPO_DIR).exists() else ['git', 'clone', REPO_URL, REPO_DIR]
subprocess.run(cmd, check=True, env=env)
del token, credential
os.chdir(REPO_DIR)
print('repo ready:', pathlib.Path.cwd())

## 2. Install dependencies

In [ ]:
!nvidia-smi
# Set True to verify training alone while HF/Xet is down. Set False before evaluation to fetch the assets.
TRAINING_ONLY = False
import os
os.environ['SKIP_LIBERO_ASSETS'] = '1' if TRAINING_ONLY else '0'
!bash scripts/setup_colab.sh

# Exports from setup_colab.sh do not carry into other cells, so set them on the runtime too
import sys
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['PYTHONPATH'] = '/content/LIBERO-plus:' + os.environ.get('PYTHONPATH', '')
if '/content/LIBERO-plus' not in sys.path:
    sys.path.insert(0, '/content/LIBERO-plus')

# Early check that LIBERO-Plus really imports and that GPU/EGL are usable
import torch, libero, lerobot
from libero.libero import benchmark
assert torch.cuda.is_available(), 'set the runtime accelerator to GPU'
suites = benchmark.get_benchmark_dict()
assert 'libero_spatial' in suites
print('GPU:', torch.cuda.get_device_name(0))
print('LeRobot:', lerobot.__version__)
print('LIBERO-Plus:', list(libero.__path__)[0])
print('setup smoke test: OK')

# Env smoke test that goes all the way to building the robosuite controller.
# Catches incompatibilities that import cleanly but break at runtime (e.g. mujoco's mj_fullM API change).
if not TRAINING_ONLY:
    from libero.libero import get_libero_path
    from libero.libero.envs import OffScreenRenderEnv
    _task = suites['libero_spatial']().get_task(0)
    _bddl = os.path.join(get_libero_path('bddl_files'), _task.problem_folder, _task.bddl_file)
    _env = OffScreenRenderEnv(bddl_file_name=_bddl, camera_heights=128, camera_widths=128)
    _env.reset()
    _env.close()
    print('env smoke test: OK')


## 2.1 One-episode smoke evaluation with the public checkpoint

Before committing to a long training run, exercise the entire path once, from model loading through MuJoCo/EGL rollout.

In [ ]:
# The public checkpoint (lerobot/smolvla_libero_plus) was trained with the camera1/camera2
# feature names (the same mapping as camera_name_mapping in upstream's train_config.json).
CAM_MAP = '{"agentview_image":"camera1","robot0_eye_in_hand_image":"camera2"}'
!lerobot-eval \
    --policy.path=lerobot/smolvla_libero_plus \
    --env.type=libero_plus \
    --env.task=libero_spatial \
    --env.task_ids='[0]' \
    --env.camera_name_mapping='{CAM_MAP}' \
    --eval.batch_size=1 \
    --eval.n_episodes=1 \
    --eval.use_async_envs=false \
    --output_dir={DRIVE_ROOT}/smoke_eval


## 3. Generate the held-out split

Episodes in the `camera_pose` category are excluded from training and diverted to held-out.
The per-episode categories come from `data/episode_categories.csv`, recovered from RLDS
(`Sylvest/libero_plus_rlds`) — see recover_episode_categories.py.


In [ ]:
!python src/make_heldout_split.py \
    --episode_categories data/episode_categories.csv \
    --heldout_categories camera_pose \
    --output_dir {DRIVE_ROOT}/splits


## 4. SFT training

Checkpoints are written to VM-local storage during training; only the latest is synced to Drive
(`outputs/checkpoints/`) at the end, to conserve Drive quota. If you would rather write straight to
Drive as insurance against a dropped session, pass `CKPT_ON_DRIVE=1`.


In [ ]:
import torch
# Upstream's lerobot/smolvla_libero_plus was trained with steps=20000, batch=32.
# save_freq=5000 keeps checkpoints on Drive, so if evaluation shows this is not enough, resume and continue.
STEPS = 20_000
gpu_name = torch.cuda.get_device_name(0)
BATCH_SIZE = 8 if 'T4' in gpu_name else 16 if 'L4' in gpu_name else 32
print(f'GPU={gpu_name} batch_size={BATCH_SIZE} steps={STEPS}')
!bash scripts/train_sft.sh {DRIVE_ROOT} {STEPS} {BATCH_SIZE}

### Training curve

Extract step/loss from `train.log` and check for NaN, divergence, or stagnation.

In [ ]:
import math, pathlib, re
import matplotlib.pyplot as plt

# lerobot-train log line: "... step:6K smpl:192K ep:24 epch:0.05 loss:0.123 grdn:... lr:..."
# step is rounded by format_big_number (>=1000 shown as K/M with 0 decimals), so
# expand K/M to recover it. The x-axis resolution drops to 1K, which is fine for checking the trend.
LOG_NAME = 'sft'  # change this if you trained with a RUN_NAME
log_path = pathlib.Path(DRIVE_ROOT) / f'logs/{LOG_NAME}.log'
text = log_path.read_text(errors='replace') if log_path.exists() else ''
UNIT = {'': 1, 'K': 1_000, 'M': 1_000_000}
pairs = [
    (int(float(s) * UNIT[u]), float(v))
    for s, u, v in re.findall(r'step:([0-9.]+)([KM]?)\b.*?loss:([0-9.eE+-]+)', text)
]
assert pairs, f'cannot extract loss from: {log_path}'
steps, losses = zip(*pairs)
assert all(math.isfinite(v) for v in losses), 'detected NaN/Inf in loss'
plt.plot(steps, losses)
plt.xlabel('step'); plt.ylabel('loss'); plt.grid(True); plt.show()
print('latest:', pairs[-10:])


## 5. Evaluation (in-dist + held-out) — exploratory

10 tasks per suite × 10 episodes × 4 suites = 400 episodes each.
Override `EVAL_TRIALS` to raise or lower the count.

> **This cell does not produce a confirmatory result.** in-dist and held-out are
> drawn from **different task pools**, so the gap between them confounds viewpoint
> sensitivity with task difficulty and cannot be read as a viewpoint-generalization
> measurement. And in LIBERO-Plus the initial state cannot be chosen with the reset
> seed (`LiberoEnv.reset()` overwrites it via
> `set_init_state(init_states[init_state_id % N])`), so evaluating an already-trained
> task does not give held-out initial conditions either. For the confirmatory method
> comparison, use the preregistered unseen-task evaluation in cell 7.

In [ ]:
CKPT = f'{DRIVE_ROOT}/outputs/checkpoints/last'
!bash scripts/eval.sh {CKPT} in_dist
!bash scripts/eval.sh {CKPT} heldout

## 6. Results summary

In [ ]:
import json, pathlib
for mode in ['in_dist', 'heldout']:
    p = pathlib.Path(f'{DRIVE_ROOT}/outputs/checkpoints/last/eval_{mode}/overall.json')
    if p.exists():
        print(f'==== {mode} ====')
        print(json.dumps(json.loads(p.read_text()), indent=2))

## 7. Preregistered unseen-task evaluation (GRPO vs SFT)

In [ ]:
# ============================================================
# Preregistered unseen-task evaluation  (GRPO vs SFT, 48 tasks x 3 eps = 288 episodes)
#   Design: data/eval/PREREGISTRATION_unseen_task_eval.md (committed before the run)
#   Run cell 2 (Drive mount + clone) and cell 4 (install dependencies) first.
# Runs in the background and writes log/status to Drive, so progress survives a VM reclaim.
# ============================================================
import os, pathlib, subprocess, textwrap

DRIVE_ROOT = '/content/drive/MyDrive/SmolVLA_RL'
REPO_DIR   = '/content/SmolVLA_RL'
SFT_CKPT   = f'{DRIVE_ROOT}/outputs/sft_base_heldout/checkpoints/last'
GRPO_CKPT  = f'{DRIVE_ROOT}/grpo_run8/ckpt_latest'
OUT        = f'{DRIVE_ROOT}/eval_unseen_prereg'   # on Drive = per-task resilience to reclamation

# Pin to the paper branch (which holds the preregistration and the scripts)
subprocess.run(['git','-C',REPO_DIR,'fetch','origin','paper'], check=True)
subprocess.run(['git','-C',REPO_DIR,'checkout','-B','paper','origin/paper'], check=True)
print(subprocess.run(['git','-C',REPO_DIR,'log','--oneline','-1'],
                     capture_output=True, text=True).stdout)

for name, ck in (('sft', SFT_CKPT), ('grpo', GRPO_CKPT)):
    assert pathlib.Path(ck).exists(), f'{name} checkpoint not found: {ck}'
    print('ok', name, ck)

os.makedirs(OUT, exist_ok=True)
env = os.environ.copy()
env.update(SFT_CKPT=SFT_CKPT, GRPO_CKPT=GRPO_CKPT, OUT=OUT,
           LIBERO_ROOT='/content/LIBERO-plus', MUJOCO_GL='egl', PYOPENGL_PLATFORM='egl')
log = f'{DRIVE_ROOT}/_unseen_prereg.log'
st  = f'{DRIVE_ROOT}/_unseen_prereg.status'
pathlib.Path(st).write_text('RUNNING\n')
cmd = (f'cd {REPO_DIR} && bash scripts/eval_unseen_prereg.sh > {log} 2>&1; '
       f'echo "EXIT=$?" >> {st}')
proc = subprocess.Popen(['bash','-lc',cmd], env=env)
print('launched pid', proc.pid, '\nlog:', log, '\nstatus:', st)


In [ ]:
# Progress polling (safe to re-run any number of times). 288 episodes should take ~45 min.
import pathlib, re
DRIVE_ROOT='/content/drive/MyDrive/SmolVLA_RL'
log=pathlib.Path(f'{DRIVE_ROOT}/_unseen_prereg.log')
st =pathlib.Path(f'{DRIVE_ROOT}/_unseen_prereg.status')
print('status:', st.read_text().strip() if st.exists() else '(none)')
if log.exists():
    lines=log.read_text().splitlines()
    done=[l for l in lines if re.search(r'\[eval\] \S+ \d+/\d+', l)]
    print(f'tasks done: {len(done)}/96 (48 tasks x 2 arms)')
    print('\n'.join(lines[-12:]))
